In [ ]:
import glob
import os
import pandas as pd


# 데이터셋 로드
admissions = pd.read_csv(r"/content/admissions.csv")
patients = pd.read_csv(r"/content/patients.csv")
icustays = pd.read_csv(r"/content/icustays.csv")
inputevents = pd.read_csv(r"/content/inputevents.csv")
outputevents = pd.read_csv(r"/content/outputevents.csv")
ingredientevents = pd.read_csv(r"/content/ingredientevents.csv")
d_item = pd.read_csv(r"/content/d_items.csv")

# 사망 및 생존 환자 모두 필터링 (필터링 없이 모든 데이터 사용)
filtered_admissions = admissions

#1. 두번 이상 사망한 환자
deceased_subjects = admissions[admissions['hospital_expire_flag'] == 1]
multiple_deaths = deceased_subjects['subject_id'].value_counts()
subject_multiple_deaths = multiple_deaths[multiple_deaths > 1].index.tolist()
print("두 번 이상 사망한 subject_id:", subject_multiple_deaths)
multiple_death_records = admissions[admissions['subject_id'].isin(subject_multiple_deaths)]
admissions = multiple_death_records

# 입원 데이터와 환자 인구통계 데이터 병합
merged_data = filtered_admissions.merge(patients, on='subject_id')

# ICU 데이터 병합
merged_data = merged_data.merge(icustays, on=['subject_id', 'hadm_id'])

# 관련 환자의 입력 이벤트 필터링
inputevents_filtered = inputevents[inputevents['subject_id'].isin(merged_data['subject_id'])]

# 치료 관련 요인 추가
merged_data = merged_data.merge(inputevents_filtered, on=['subject_id', 'hadm_id'], how='left')


In [ ]:
outputevents_data = pd.merge(outputevents, d_item[['itemid', 'label', 'category']], on='itemid', how='left')
outputevents_data.drop(columns='itemid', inplace=True)
outputevents_data

In [ ]:
merged_data = merged_data[merged_data['subject_id'] != 10676055]
merged_data

In [ ]:
def age_group(age):
    if age <= 30:
        return '0-30'
    elif age <= 60:
        return '31-60'
    elif age <= 90:
        return '61-90'
    else:
        return '90+'

# 연령대 컬럼 추가
merged_data['age_group'] = merged_data['anchor_age'].apply(age_group)

# 인종을 범주형 데이터로 변환
merged_data['race'] = merged_data['race'].astype('category')


# 그룹화된 데이터 출력
merged_data


In [ ]:
# merged_data가 DataFrame이라고 가정합니다.

# 'endtime' 및 'starttime'를 datetime 객체로 변환합니다.
merged_data['endtime'] = pd.to_datetime(merged_data['endtime'])
merged_data['starttime'] = pd.to_datetime(merged_data['starttime'])

merged_data['admittime'] = pd.to_datetime(merged_data['admittime'])
merged_data['dischtime'] = pd.to_datetime(merged_data['dischtime'])
merged_data['deathtime'] = pd.to_datetime(merged_data['deathtime'], errors='coerce')  # Handle NaT if deathtime is missing

# Calculate 'hospital_stay' as the difference between dischtime and admittime (in days)
merged_data['hospital_stay'] = (merged_data['dischtime'] - merged_data['admittime']).dt.total_seconds() / (60 * 60 * 24)

# Calculate 'time_until_death' as the difference between deathtime and admittime (in days)
merged_data['time_until_death'] = (merged_data['deathtime'] - merged_data['admittime']).dt.total_seconds() / (60 * 60 * 24)

merged_data = merged_data[(merged_data['time_until_death'] >= 0) | (merged_data['time_until_death'].isna())]

# 'hospital_stay' 컬럼에서 결측값은 유지하고 음수인 값만 제거
merged_data = merged_data[(merged_data['hospital_stay'] >= 0) | (merged_data['hospital_stay'].isna())]

merged_data

In [ ]:
merged_data = merged_data.dropna(subset=[
    'patientweight',
    'starttime',
    'endtime',
    'itemid',
    'amount',
    'amountuom'
])

In [ ]:

#2. 안죽었는데 죽은 데이터(deathtime nan일 때 hospital_expire_flag = 1)
print("안죽었는데 죽은 데이터:")
nan_death_records = merged_data[(merged_data['deathtime'].isna()) & (merged_data['hospital_expire_flag'] == 1)]
nan_death_records

merged_data = merged_data[~(merged_data['deathtime'].isna() & (merged_data['hospital_expire_flag'] == 1))]

# prompt: merged_data에서 multiple_death_records를 뺀 데이터 프레임만 출력해줘

# multiple_death_records를 제외한 데이터프레임 생성
merged_data = merged_data[~merged_data['subject_id'].isin(subject_multiple_deaths)]

merged_data = merged_data.drop(columns=[
    'admittime',
    'dischtime',
    'outtime',
    'deathtime',
    'starttime',
    'endtime',
    'intime',
])

merged_data.drop(columns=['rate', 'rateuom'], inplace=True)

In [ ]:
# prompt: merged_data의 d_item을  itemid를 기준으로 합쳐주는데, label, category만 넣어줘

# d_item과 merged_data를 itemid를 기준으로 병합합니다.
# label과 category만 merged_data에 추가합니다.
preprocessing_data = pd.merge(merged_data, d_item[['itemid', 'label', 'category']], on='itemid', how='left')
preprocessing_data.drop(columns='itemid', inplace=True)

In [ ]:
preprocessing_data.rename(columns={'label': 'inputevent_label'}, inplace=True)

In [ ]:
preprocessing_data

In [ ]:
import zipfile

# preprocessing_data를 preprocessing_data(fixed).zip으로 저장
zip_filename = 'preprocessing_data(fixed).zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    preprocessing_data.to_csv('preprocessing_data.csv', index=False)  # CSV 파일로 저장
    zipf.write('preprocessing_data.csv')

# zip 파일을 content 폴더로 이동 (필요시)
!mv preprocessing_data(fixed).zip /content/

성별

In [ ]:
import pandas as pd
import zipfile
import os

# 경로 설정
zip_path_fixed = '/content/preprocessing_data(fixed).zip'
extract_path_fixed = '/content/preprocessing_fixed_data/'

# ZIP 파일 추출
if os.path.exists(zip_path_fixed):
    try:
        with zipfile.ZipFile(zip_path_fixed, 'r') as zip_ref:
            zip_ref.extractall(extract_path_fixed)
        print("Fixed ZIP file successfully extracted.")
    except zipfile.BadZipFile:
        print("Error: Not a valid ZIP file.")
    except Exception as e:
        print(f"Error: {e}")
else:
    print("Error: ZIP file not found.")

# 추출된 파일 확인
if os.path.exists(extract_path_fixed):
    extracted_files_fixed = os.listdir(extract_path_fixed)
    print("Extracted files:")
    print(extracted_files_fixed)
else:
    print(f"Error: Extract directory not found: {extract_path_fixed}")
    os.makedirs(extract_path_fixed)
    print(f"Directory created: {extract_path_fixed}")
    extracted_files_fixed = []

# CSV 파일 로드
csv_file_path_fixed = os.path.join(extract_path_fixed, 'preprocessing_data.csv')
if 'preprocessing_data.csv' in extracted_files_fixed:
    try:
        data_fixed = pd.read_csv(csv_file_path_fixed)
        print("Fixed preprocessing CSV file successfully loaded.")
    except pd.errors.EmptyDataError:
        print("Error: Fixed preprocessing CSV file is empty.")
        data_fixed = pd.DataFrame()
    except Exception as e:
        print(f"Error: {e}")
        data_fixed = pd.DataFrame()
else:
    print("Error: 'preprocessing_data.csv' file not found.")
    data_fixed = pd.DataFrame()

# 데이터 분석 및 성별 비율 계산
if not data_fixed.empty:
    if 'gender' in data_fixed.columns and 'hospital_expire_flag' in data_fixed.columns:
        try:
            survival_gender_ratio_fixed = (
                data_fixed.groupby(['hospital_expire_flag', 'gender']).size()
                / data_fixed.groupby('hospital_expire_flag').size()
            )
            survival_gender_ratio_fixed = survival_gender_ratio_fixed.unstack()
            print("Gender ratio for each survival status:")
            print(survival_gender_ratio_fixed)
        except Exception as e:
            print(f"Error during gender ratio analysis: {e}")
    else:
        print("Error: 'gender' or 'hospital_expire_flag' column not found in data.")
else:
    print("Error: Fixed preprocessing data is empty.")


In [ ]:
import matplotlib.pyplot as plt

if 'F' in survival_gender_ratio_fixed.columns and 'M' in survival_gender_ratio_fixed.columns:
    survival_gender_ratio_fixed.plot(kind='bar', figsize=(10, 6))

    plt.title("Gender Ratio by Survival Status")
    plt.xlabel("Hospital Expire Flag (0: Survived, 1: Deceased)")
    plt.ylabel("Proportion")
    plt.legend(title="Gender", labels=["Female", "Male"])
    plt.xticks(rotation=0)
    plt.tight_layout()

    plt.show()


인종

In [ ]:

if 'race' in data_fixed.columns and 'hospital_expire_flag' in data_fixed.columns:
    try:

        race_survival_ratio = (
            data_fixed.groupby(['hospital_expire_flag', 'race']).size()
            / data_fixed.groupby('hospital_expire_flag').size()
        )
        race_survival_ratio = race_survival_ratio.unstack()

        print("Race ratio for each survival status:")
        print(race_survival_ratio)
    except Exception as e:
        print(f"Error during race ratio analysis: {e}")
else:
    print("Error: 'race' or 'hospital_expire_flag' column not found in data.")


In [ ]:

race_survival_ratio.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 8),
    color=plt.cm.tab10.colors,
    alpha=0.85
)

plt.title("Stacked Bar Plot of Race Ratio by Survival Status", fontsize=16)
plt.xlabel("Hospital Expire Flag (0: Survived, 1: Deceased)", fontsize=14)
plt.ylabel("Proportion", fontsize=14)
plt.legend(title="Race", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0, fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 파일 경로 설정
file_path1 = r"/content/admissions.csv"

# 데이터 읽기
try:
    admissions = pd.read_csv(file_path1)
    print("File successfully loaded.")
except FileNotFoundError:
    print(f"Error: File not found at {file_path1}")
except pd.errors.EmptyDataError:
    print("Error: The file is empty.")
except Exception as e:
    print(f"Error while loading the file: {e}")

# 중복 제거
admissions = admissions.drop_duplicates(subset='subject_id', keep='first')

# 병원 사망 상태 및 인종에 따른 데이터 집계
race_flag_counts = admissions.groupby(['race', 'hospital_expire_flag']).size().reset_index(name='count')

# 인종별 총합 계산 및 비율 추가
race_totals = race_flag_counts.groupby('race')['count'].transform('sum')
race_flag_counts['proportion'] = race_flag_counts['count'] / race_totals

# 피벗 테이블 생성
pivot_data = race_flag_counts.pivot(index='race', columns='hospital_expire_flag', values='proportion').fillna(0)

# 시각화
plt.figure(figsize=(15, 7))

# 색상 설정
colors = {0: 'purple', 1: 'yellow'}
bottom_values = None

# 스택형 막대 그래프 생성
for flag in pivot_data.columns:
    plt.bar(
        pivot_data.index,
        pivot_data[flag],
        bottom=bottom_values,
        label=f'Hospital Expire Flag {flag}',
        color=colors[flag]
    )
    if bottom_values is None:
        bottom_values = pivot_data[flag]
    else:
        bottom_values += pivot_data[flag]

# 그래프 제목 및 라벨 설정
plt.title('Proportion of Patients by Race and Expiration Status (Stacked)')
plt.xlabel('Race')
plt.ylabel('Proportion')
plt.xticks(rotation=90, ha='center')
plt.legend(title='Hospital Expire Flag')

# 레이아웃 조정 및 출력
plt.tight_layout()
plt.show()


나이

In [ ]:

if 'anchor_age' in data_fixed.columns and 'hospital_expire_flag' in data_fixed.columns:
    try:

        data_fixed['age_group'] = pd.cut(data_fixed['anchor_age'], bins=range(0, 101, 10), right=False, labels=[f"{i}-{i+9}" for i in range(0, 100, 10)])


        age_survival_ratio = (
            data_fixed.groupby(['hospital_expire_flag', 'age_group']).size()
            / data_fixed.groupby('hospital_expire_flag').size()
        )
        age_survival_ratio = age_survival_ratio.unstack()

        print("Age ratio by survival status:")
        for status in age_survival_ratio.index:
            status_name = "Survived (0)" if status == 0 else "Deceased (1)"
            print(f"\n{status_name}:")
            for age_group, ratio in age_survival_ratio.loc[status].items():
                print(f"  {age_group}: {ratio:.2%}")
    except Exception as e:
        print(f"Error during age ratio analysis: {e}")
else:
    print("Error: 'anchor_age' or 'hospital_expire_flag' column not found in data.")


In [ ]:
import matplotlib.pyplot as plt

if 'age_group' in data_fixed.columns and 'hospital_expire_flag' in data_fixed.columns:
    try:

        age_survival_ratio = (
            data_fixed.groupby(['hospital_expire_flag', 'age_group']).size()
            / data_fixed.groupby('hospital_expire_flag').size()
        )
        age_survival_ratio = age_survival_ratio.unstack()

        age_survival_ratio.plot(kind='bar', figsize=(12, 8))

        plt.title("Age Ratio by Survival Status")
        plt.xlabel("Hospital Expire Flag (0: Survived, 1: Deceased)")
        plt.ylabel("Proportion")
        plt.legend(title="Age Group")
        plt.xticks(rotation=0)
        plt.tight_layout()

        plt.show()
    except Exception as e:
        print(f"Error during age ratio visualization: {e}")
else:
    print("Error: 'age_group' or 'hospital_expire_flag' column not found in data.")


인종별 인구수와 인종별 사망자수 확인 -> 인종비율이 달라 유의미한 결과 아니라고 판단


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"

icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)


admissions['admittime'] = pd.to_datetime(admissions['admittime'])
admissions['dischtime'] = pd.to_datetime(admissions['dischtime'])



admissions = admissions.sort_values(by=['subject_id', 'admittime'])

admissions['is_last_admission'] = admissions.groupby('subject_id')['admittime'].transform('max') == admissions['admittime']
previous_admissions = admissions[~admissions['is_last_admission']]

previous_summary = previous_admissions.groupby('subject_id').agg(
    total_previous_admissions=('hadm_id', 'count'),
avg_previous_los=('dischtime', lambda x: ((x - admissions.loc[x.index, 'admittime']).dt.total_seconds()).mean() / 3600)
).reset_index()

last_admissions = admissions[admissions['is_last_admission']]
merged_data = pd.merge(last_admissions, previous_summary, on='subject_id', how='left')

icustays_aggregated = icustays.groupby(['subject_id', 'hadm_id']).agg(
    avg_icu_los=('los', 'mean'),
    icu_stay_count=('subject_id', 'count')
).reset_index()
merged_data = pd.merge(merged_data, icustays_aggregated, on=['subject_id', 'hadm_id'], how='left')
merged_data = pd.merge(merged_data, patients, on='subject_id', how='left')

if 'race' in merged_data.columns and 'hospital_expire_flag' in merged_data.columns:

    population_by_race = merged_data['race'].value_counts().reset_index()
    population_by_race.columns = ['race', 'population']

    deaths_by_race = merged_data[merged_data['hospital_expire_flag'] == 1]['race'].value_counts().reset_index()
    deaths_by_race.columns = ['race', 'deaths']

    race_summary = pd.merge(population_by_race, deaths_by_race, on='race', how='left')

    race_summary['deaths'].fillna(0, inplace=True)
    race_summary['deaths'] = race_summary['deaths'].astype(int)

    print("Population and Deaths by Race:")
    print(race_summary)
else:
    print("The 'race' or 'hospital_expire_flag' columns are missing in the admissions dataset.")

사망자들에게 가장 많이 시행된 procedure top 10

In [ ]:
file_path10 = r"/content/procedureevents.csv"
file_path11 = r"/content/d_items.csv"
procedureevents = pd.read_csv(file_path10)
d_items = pd.read_csv(file_path11)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()


admissions_summary = admissions.groupby("subject_id").agg(
    total_admissions=("hadm_id", "count"),
    last_admittime=("admittime", "max")
).reset_index()

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions[['subject_id', 'hadm_id', 'hospital_expire_flag']], on="subject_id", how="left")
)


deceased_patients = merged_data[merged_data['hospital_expire_flag'] == 1]

deceased_procedures = pd.merge(
    deceased_patients[['subject_id', 'hadm_id']],
    procedureevents,
    on=['subject_id', 'hadm_id'],
    how='inner'
)

deceased_procedures_with_names = pd.merge(
    deceased_procedures,
    d_items[['itemid', 'label']],
    left_on='itemid',
    right_on='itemid',
    how='left'
)

most_common_procedures = deceased_procedures_with_names['label'].value_counts().reset_index()
most_common_procedures.columns = ['procedure_name', 'count']

print(most_common_procedures.head(10))
plt.figure(figsize=(10, 6))
plt.barh(
    most_common_procedures['procedure_name'].head(10),
    most_common_procedures['count'].head(10),
    align='center'
)
plt.xlabel('Count')
plt.ylabel('Procedure Name')
plt.title('Top 10 Most Common Procedures for Deceased Patients')
plt.gca().invert_yaxis()
plt.show()


생존자들에게 가장 많이 시행된 procedure top 10  

In [ ]:

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()


admissions_summary = admissions.groupby("subject_id").agg(
    total_admissions=("hadm_id", "count"),
    last_admittime=("admittime", "max")
).reset_index()

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions[['subject_id', 'hadm_id', 'hospital_expire_flag']], on="subject_id", how="left")
)

deceased_patients = merged_data[merged_data['hospital_expire_flag'] == 0]

deceased_procedures = pd.merge(
    deceased_patients[['subject_id', 'hadm_id']],
    procedureevents,
    on=['subject_id', 'hadm_id'],
    how='inner'
)

deceased_procedures_with_names = pd.merge(
    deceased_procedures,
    d_items[['itemid', 'label']],
    left_on='itemid',
    right_on='itemid',
    how='left'
)

most_common_procedures = deceased_procedures_with_names['label'].value_counts().reset_index()
most_common_procedures.columns = ['procedure_name', 'count']

print(most_common_procedures.head(10))

plt.figure(figsize=(10, 6))
plt.barh(
    most_common_procedures['procedure_name'].head(10),
    most_common_procedures['count'].head(10),
    align='center'
)
plt.xlabel('Count')
plt.ylabel('Procedure Name')
plt.title('Top 10 Most Common Procedures for Alive Patients')
plt.gca().invert_yaxis()
plt.show()


사망자들에게 가장 많이 주입된 약물 성분 top10

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path11 = r"/content/ingredientevents.csv"
file_path7 = r"/content/d_items.csv"


icustays = pd.read_csv(file_path2)
ingredientevents = pd.read_csv(file_path11)
patients = pd.read_csv(file_path4)
d_items = pd.read_csv(file_path7)


icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()


admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")


admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]


merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

deceased_patients = merged_data[merged_data['hospital_expire_flag'] == 1][['subject_id']]


deceased_ingredientevents = pd.merge(
    deceased_patients,
    ingredientevents,
    on='subject_id',
    how='inner'
)

deceased_item_counts = deceased_ingredientevents['itemid'].value_counts().reset_index()
deceased_item_counts.columns = ['itemid', 'count']

deceased_item_counts = pd.merge(
    deceased_item_counts,
    d_items[['itemid', 'label']],
    on='itemid',
    how='left'
)

colors_list = ['lightpink', 'lightblue', 'lightgreen', 'lightyellow', 'lightcoral', 'lightsalmon', 'lightseagreen', 'lightgoldenrodyellow', 'lightskyblue', 'lightcyan']

plt.figure(figsize=(12, 8))
bars = plt.barh(
    deceased_item_counts['label'],
    deceased_item_counts['count'],
    align='center',
    color=colors_list * (len(deceased_item_counts) // len(colors_list)) + colors_list[:len(deceased_item_counts) % len(colors_list)]  # 색상 순환
)

plt.xlabel('Count', fontsize=12)
plt.ylabel('Ingredient Name', fontsize=12)
plt.title('Most Frequent Ingredient Events for Deceased Patients', fontsize=15, fontweight='bold')

plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(width + 1, bar.get_y() + bar.get_height() / 2,
             str(width), va='center', ha='left', fontsize=10, color='black', fontweight='bold')

plt.grid(True, axis='x', linestyle='--', alpha=0.7)



plt.tight_layout()

plt.show()


생존자들에게 가장 많이 주입된 약물 성분 top 10

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path11 = r"/content/ingredientevents.csv"
file_path7 = r"/content/d_items.csv"

icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)
d_items = pd.read_csv(file_path7)
ingredientevents = pd.read_csv(file_path11)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

alive_patients = merged_data[merged_data['hospital_expire_flag'] == 0][['subject_id']]

alive_ingredientevents = pd.merge(
    alive_patients,
    ingredientevents,
    on='subject_id',
    how='inner'
)


alive_item_counts = alive_ingredientevents['itemid'].value_counts().reset_index()
alive_item_counts.columns = ['itemid', 'count']

alive_item_counts = pd.merge(
    alive_item_counts,
    d_items[['itemid', 'label']],
    on='itemid',
    how='left'
)

top_10_alive_items = alive_item_counts.head(10)

colors_list = ['lightpink', 'lightblue', 'lightgreen', 'lightyellow', 'lightcoral', 'lightsalmon', 'lightseagreen', 'lightgoldenrodyellow', 'lightskyblue', 'lightcyan']

plt.figure(figsize=(12, 8))
bars = plt.barh(
    top_10_alive_items['label'],
    top_10_alive_items['count'],
    align='center',
    color=colors_list * (len(top_10_alive_items) // len(colors_list)) + colors_list[:len(top_10_alive_items) % len(colors_list)]  # 색상 순환
)

plt.xlabel('Count', fontsize=12)
plt.ylabel('Ingredient Name', fontsize=12)
plt.title('Top 10 Most Frequent Ingredient Events for Alive Patients', fontsize=15, fontweight='bold')

plt.gca().invert_yaxis()

for bar in bars:
    width = bar.get_width()
    plt.text(width + 1, bar.get_y() + bar.get_height() / 2,
             str(width), va='center', ha='left', fontsize=10, color='black', fontweight='bold')


plt.grid(True, axis='x', linestyle='--', alpha=0.7)


plt.tight_layout()

plt.show()


생존자들에게 가장 많이 주입된 약물 top 10

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path8 = r"/content/inputevents.csv"
file_path9 = r"/content/chartevents.csv"

icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)
inputevents = pd.read_csv(file_path8)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions_summary = (
    admissions.sort_values(by=["subject_id", "admittime"])
              .groupby("subject_id")
              .last()
              .reset_index()
)
admissions_summary["admittime"] = pd.to_datetime(admissions_summary["admittime"])


merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary[["subject_id", "hadm_id", "admittime", "hospital_expire_flag"]],
                   on="subject_id", how="left")
)

inputevents_with_flag = pd.merge(
    inputevents,
    merged_data[["subject_id", "hospital_expire_flag"]],
    on="subject_id",
    how="inner"
)


inputevents_filtered = inputevents_with_flag[inputevents_with_flag["hospital_expire_flag"] == 0]


itemid_counts_filtered = inputevents_filtered["itemid"].value_counts().reset_index()
itemid_counts_filtered.columns = ["itemid", "count"]
itemid_with_labels_filtered = pd.merge(itemid_counts_filtered, d_items, on="itemid", how="left")


top_10_filtered_items = itemid_with_labels_filtered.head(10)


colors = ['#B56576', '#6D6875', '#355070', '#6A994E', '#A393BF',
          '#DAB785', '#B5838D', '#8A817C', '#E5989B', '#AA4465']


plt.figure(figsize=(10, 6))
plt.bar(
    top_10_filtered_items['label'],
    top_10_filtered_items['count'],
    color=colors
)
plt.xlabel('Ingredient Name', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Top 10 Most Frequent Inputevents Events for Alive Patients', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


사망자들에게 가장 많이 주입된 약물 top 10

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path8 = r"/content/inputevents.csv"
file_path9 = r"/content/chartevents.csv"



icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)
inputevents = pd.read_csv(file_path8)
chartevents = pd.read_csv(file_path9)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions_summary = (
    admissions.sort_values(by=["subject_id", "admittime"])
              .groupby("subject_id")
              .last()
              .reset_index()
)
admissions_summary["admittime"] = pd.to_datetime(admissions_summary["admittime"])

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary[["subject_id", "hadm_id", "admittime", "hospital_expire_flag"]],
                   on="subject_id", how="left")
)


merged_data["admittime"] = pd.to_datetime(merged_data["admittime"], errors="coerce")

inputevents_with_flag = pd.merge(
    inputevents,
    merged_data[["subject_id", "hospital_expire_flag"]],
    on="subject_id",
    how="inner"
)

inputevents_filtered = inputevents_with_flag[inputevents_with_flag["hospital_expire_flag"] == 1]

itemid_counts_filtered = inputevents_filtered["itemid"].value_counts().reset_index()
itemid_counts_filtered.columns = ["itemid", "count"]

itemid_with_labels_filtered = pd.merge(itemid_counts_filtered, d_items, on="itemid", how="left")

top_10_filtered_items = itemid_with_labels_filtered.head(10)
print(top_10_filtered_items)


colors = ['#B56576', '#6D6875', '#355070', '#6A994E', '#A393BF',
          '#DAB785', '#B5838D', '#8A817C', '#E5989B', '#AA4465']


plt.figure(figsize=(10, 6))
plt.bar(
    top_10_filtered_items['label'],
    top_10_filtered_items['count'],
    color=colors
)
plt.xlabel('Ingredient Name', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Top 10 Most Frequent Inputevents for Deceased Patients', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



약물 별 사망자 vs 생존자 비율 비교

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt


icustays_path = "/content/icustays.csv"
patients_path = "/content/patients.csv"
inputevents_path = "/content/inputevents.csv"
d_items_path = "/content/d_items.csv"


icustays = pd.read_csv(icustays_path)
patients = pd.read_csv(patients_path)
inputevents = pd.read_csv(inputevents_path)
d_items = pd.read_csv(d_items_path)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions_summary = (
    admissions.sort_values(by=["subject_id", "admittime"])
    .groupby("subject_id").last().reset_index()
)

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
    .merge(admissions_summary[["subject_id", "hadm_id", "admittime", "hospital_expire_flag"]],
           on="subject_id", how="left")
)

inputevents_with_flag = pd.merge(
    inputevents,
    merged_data[["subject_id", "hospital_expire_flag"]],
    on="subject_id",
    how="inner"
)

itemid_counts = inputevents["itemid"].value_counts().reset_index()
itemid_counts.columns = ["itemid", "count"]

top_10_itemids = itemid_counts.head(10)

alive_counts = inputevents_with_flag[inputevents_with_flag["hospital_expire_flag"] == 0]
alive_counts = alive_counts["itemid"].value_counts().reset_index()
alive_counts.columns = ["itemid", "alive_count"]

deceased_counts = inputevents_with_flag[inputevents_with_flag["hospital_expire_flag"] == 1]
deceased_counts = deceased_counts["itemid"].value_counts().reset_index()
deceased_counts.columns = ["itemid", "deceased_count"]

combined_counts = pd.merge(top_10_itemids, alive_counts, on="itemid", how="left").merge(
    deceased_counts, on="itemid", how="left"
).fillna(0)

labels = pd.merge(combined_counts, d_items[["itemid", "label"]], on="itemid", how="left")["label"]
alive = combined_counts["alive_count"]
deceased = combined_counts["deceased_count"]

color_alive = '#6AA84F'
color_deceased = '#3C78D8'

plt.figure(figsize=(12, 8))
bar1 = plt.bar(labels, alive, label="Alive Patients", color=color_alive)
bar2 = plt.bar(labels, deceased, bottom=alive, label="Deceased Patients", color=color_deceased)

plt.xlabel("Input Event Name", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Top 10 Inputevents: Deceased vs. Alive Patients", fontsize=14)
plt.xticks(rotation=45, ha="right")

plt.legend(title="Patient Status", fontsize=10)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()



사망자 vs 생존자 노르에피네프린 사용량 총합 비교

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


icustays = pd.read_csv("/content/icustays.csv")
patients = pd.read_csv("/content/patients.csv")
inputevents = pd.read_csv("/content/inputevents.csv")
d_items = pd.read_csv("/content/d_items.csv")

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions_summary = (
    admissions.sort_values(by=["subject_id", "admittime"])
              .groupby("subject_id")
              .last()
              .reset_index()
)
admissions_summary["admittime"] = pd.to_datetime(admissions_summary["admittime"])

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary[["subject_id", "hadm_id", "admittime", "hospital_expire_flag"]],
                   on="subject_id", how="left")
)

inputevents_with_flag = pd.merge(
    inputevents,
    merged_data[["subject_id", "hospital_expire_flag"]],
    on="subject_id",
    how="inner"
)

inputevents_with_flag = pd.merge(
    inputevents_with_flag,
    d_items[["itemid", "label"]],
    on="itemid",
    how="left"
)


norepinephrine_deceased = inputevents_with_flag[
    (inputevents_with_flag["hospital_expire_flag"] == 1) &
    (inputevents_with_flag["label"].str.contains("norepinephrine", case=False, na=False))
]

norepinephrine_alive = inputevents_with_flag[
    (inputevents_with_flag["hospital_expire_flag"] == 0) &
    (inputevents_with_flag["label"].str.contains("norepinephrine", case=False, na=False))
]

total_norepinephrine_deceased = norepinephrine_deceased["amount"].sum()
total_norepinephrine_alive = norepinephrine_alive["amount"].sum()

labels = ["Deceased Patients", "Alive Patients"]
sizes = [total_norepinephrine_deceased, total_norepinephrine_alive]
colors = ['#F2C6AE', '#A8DADC']
explode = [0.1, 0]
plt.figure(figsize=(10, 6))
plt.pie(
    sizes,
    explode=explode,
    labels=None,
    autopct='%1.1f%%',
    startangle=90,
    shadow=True,
    colors=colors,
    pctdistance=0.85,
    wedgeprops={'edgecolor': 'black'}
)


plt.title('Comparison of Total Norepinephrine Dose (Deceased vs. Alive)', y=1.05, fontsize=15)


plt.legend(
    labels=labels,
    loc='upper left',
    fontsize=10,
    title='Patient Status'
)

plt.axis('equal')

plt.show()
plt.clf()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path8 = r"/content/outputevents.csv"
file_path9 = r"/content/d_items.csv"

icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)
outputevents = pd.read_csv(file_path8)
d_items = pd.read_csv(file_path9)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions_summary = admissions.groupby("subject_id").agg(
    total_admissions=("hadm_id", "count"),
    last_admittime=("admittime", "max")
).reset_index()

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions[['subject_id', 'hadm_id', 'hospital_expire_flag']], on="subject_id", how="left")
)

deceased_patients = merged_data[merged_data['hospital_expire_flag'] == 1][['subject_id']]

deceased_outputevents = pd.merge(
    deceased_patients,
    outputevents,
    on='subject_id',
    how='inner'
)

item_id_counts = deceased_outputevents['itemid'].value_counts().reset_index()
item_id_counts.columns = ['itemid', 'count']

item_id_counts = pd.merge(
    item_id_counts,
    d_items[['itemid', 'label']],
    on='itemid',
    how='left'
)

top_10_items = item_id_counts.head(10)
print(top_10_items)

colors = ['#FF6F61', '#6B5B95', '#88B04B', '#F7CAC9', '#92A8D1', '#034F84', '#F7786B', '#DEAB8A', '#79C753', '#CE3175']

plt.figure(figsize=(10, 6))
plt.barh(
    top_10_items['label'],
    top_10_items['count'],
    color=colors,
    align='center'
)

# 전체 count의 합계 계산
total_count = top_10_items['count'].sum()

# 각 막대의 끝에 퍼센트 표시
for index, value in enumerate(top_10_items['count']):
    percentage = (value / total_count) * 100
    plt.text(value, index, f'{percentage:.2f}%', va='center', fontsize=10)

plt.xlabel('Log Count', fontsize=12)
plt.xscale('log')
plt.ylabel('Item Name', fontsize=12)
plt.title('Top 10 Most Frequent Outputevents Items for Deceased Patients (Logarithmic Scale)', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path8 = r"/content/outputevents.csv"
file_path9 = r"/content/d_items.csv"

admissions = pd.read_csv(file_path1)
icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)
outputevents = pd.read_csv(file_path8)
d_items = pd.read_csv(file_path9)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()


admissions_summary = admissions.groupby("subject_id").agg(
    total_admissions=("hadm_id", "count"),
    last_admittime=("admittime", "max")
).reset_index()


merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions[['subject_id', 'hadm_id', 'hospital_expire_flag']], on="subject_id", how="left")
)


alive_patients = merged_data[merged_data['hospital_expire_flag'] == 0][['subject_id']]

alive_outputevents = pd.merge(
    alive_patients,
    outputevents,
    on='subject_id',
    how='inner'
)


item_id_counts = alive_outputevents['itemid'].value_counts().reset_index()
item_id_counts.columns = ['itemid', 'count']

item_id_counts = pd.merge(
    item_id_counts,
    d_items[['itemid', 'label']],
    on='itemid',
    how='left'
)

top_10_items = item_id_counts.head(10)
print(top_10_items)


colors = ['#FF6F61', '#6B5B95', '#88B04B', '#F7CAC9', '#92A8D1', '#034F84', '#F7786B', '#DEAB8A', '#79C753', '#CE3175']

plt.figure(figsize=(10, 6))
plt.barh(
    top_10_items['label'],
    top_10_items['count'],
    color=colors,
    align='center'
)
plt.xlabel('Log Count', fontsize=12)
plt.xscale('log')
plt.ylabel('Item Name', fontsize=12)
plt.title('Top 10 Most Frequent Outputevents Items for Alive Patients (Logarithmic Scale)', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path8 = r"/content/outputevents.csv"
file_path9 = r"/content/d_items.csv"

icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)
outputevents = pd.read_csv(file_path8)
d_items = pd.read_csv(file_path9)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions_summary = admissions.groupby("subject_id").agg(
    total_admissions=("hadm_id", "count"),
    last_admittime=("admittime", "max")
).reset_index()

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions[['subject_id', 'hadm_id', 'hospital_expire_flag']], on="subject_id", how="left")
)

deceased_patients = merged_data[merged_data['hospital_expire_flag'] == 1][['subject_id']]

deceased_outputevents = pd.merge(
    deceased_patients,
    outputevents,
    on='subject_id',
    how='inner'
)

item_id_counts = deceased_outputevents['itemid'].value_counts().reset_index()
item_id_counts.columns = ['itemid', 'count']

item_id_counts = pd.merge(
    item_id_counts,
    d_items[['itemid', 'label']],
    on='itemid',
    how='left'
)

top_10_items = item_id_counts.head(10)
print(top_10_items)

colors = ['#FF6F61', '#6B5B95', '#88B04B', '#F7CAC9', '#92A8D1', '#034F84', '#F7786B', '#DEAB8A', '#79C753', '#CE3175']

plt.figure(figsize=(10, 6))
plt.barh(
    top_10_items['label'],
    top_10_items['count'],
    color=colors,
    align='center'
)

# 전체 count의 합계 계산
total_count = top_10_items['count'].sum()

# 각 막대의 끝에 퍼센트 표시
for index, value in enumerate(top_10_items['count']):
    percentage = (value / total_count) * 100
    plt.text(value, index, f'{percentage:.2f}%', va='center', fontsize=10)

plt.xlabel('Log Count', fontsize=12)
plt.xscale('log')
plt.ylabel('Item Name', fontsize=12)
plt.title('Top 10 Most Frequent Outputevents Items for Deceased Patients (Logarithmic Scale)', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# 추가된 코드: 살아 있는 환자의 데이터 분석
alive_patients = merged_data[merged_data['hospital_expire_flag'] == 0][['subject_id']]

alive_outputevents = pd.merge(
    alive_patients,
    outputevents,
    on='subject_id',
    how='inner'
)

item_id_counts_alive = alive_outputevents['itemid'].value_counts().reset_index()
item_id_counts_alive.columns = ['itemid', 'count']

item_id_counts_alive = pd.merge(
    item_id_counts_alive,
    d_items[['itemid', 'label']],
    on='itemid',
    how='left'
)

top_10_items_alive = item_id_counts_alive.head(10)
print(top_10_items_alive)

plt.figure(figsize=(10, 6))
plt.barh(
    top_10_items_alive['label'],
    top_10_items_alive['count'],
    color=colors,
    align='center'
)

# 전체 count의 합계 계산 (살아 있는 환자)
total_count_alive = top_10_items_alive['count'].sum()

# 각 막대의 끝에 퍼센트 표시 (살아 있는 환자)
for index, value in enumerate(top_10_items_alive['count']):
    percentage = (value / total_count_alive) * 100
    plt.text(value, index, f'{percentage:.2f}%', va='center', fontsize=10)

plt.xlabel('Log Count', fontsize=12)
plt.xscale('log')
plt.ylabel('Item Name', fontsize=12)
plt.title('Top 10 Most Frequent Outputevents Items for Alive Patients (Logarithmic Scale)', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path9 = r"/content/chartevents.csv"



icustays = pd.read_csv(file_path2)
patients = pd.read_csv(file_path4)
chartevents = pd.read_csv(file_path9)


icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()

admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]



merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)
file_saved = r"C:\Users\ilir_\Documents\KakaoTalk Downloads\preprocessing_data(fixed)\final_combined_data.csv"
merged_data.to_csv(file_saved, index=False)

merged_data["admittime"] = pd.to_datetime(merged_data["admittime"], errors="coerce")
merged_data["dischtime"] = pd.to_datetime(merged_data["dischtime"], errors="coerce")

last_admissions = merged_data.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()

heart_rate_data = chartevents[chartevents["itemid"] == 220045]
heart_rate_data["charttime"] = pd.to_datetime(heart_rate_data["charttime"], errors="coerce")
heart_rate_data["value"] = pd.to_numeric(heart_rate_data["value"], errors="coerce")


merged_heart_rate = pd.merge(
    heart_rate_data,
    last_admissions[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]],
    on=["subject_id", "hadm_id"],
    how="inner"
)

heart_rate_last_admission = merged_heart_rate[
    (merged_heart_rate["charttime"] >= merged_heart_rate["admittime"]) &
    (merged_heart_rate["charttime"] <= merged_heart_rate["dischtime"])
]

heart_rate_summary = heart_rate_last_admission.groupby(["subject_id", "hospital_expire_flag"]).agg(
    avg_heart_rate=("value", "mean"),
    max_heart_rate=("value", "max"),
    min_heart_rate=("value", "min")
).reset_index()


import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(data=heart_rate_summary, x="hospital_expire_flag", y="avg_heart_rate")
plt.title("Average Heart Rate by Mortality (Last Hospital Admission)")
plt.xlabel("Mortality (0 = Alive, 1 = Deceased)")
plt.ylabel("Average Heart Rate (bpm)")
plt.xticks([0, 1], labels=["Alive", "Deceased"])
plt.tight_layout()
plt.show()


Q1 = heart_rate_data["value"].quantile(0.25)
Q3 = heart_rate_data["value"].quantile(0.75)
IQR = Q3 - Q1


lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR


outliers = heart_rate_data[(heart_rate_data["value"] < lower_bound) | (heart_rate_data["value"] > upper_bound)]

num_outliers = outliers.shape[0]

print(f"Number of outliers: {num_outliers}")




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


file2 = r"/content/icustays.csv"
file3 = r"/content/patients.csv"
file4 = r"/content/d_items.csv"
file5 = r"/content/chartevents.csv"


icustays = pd.read_csv(file2)
patients = pd.read_csv(file3)
d_items = pd.read_csv(file4)
chartevents = pd.read_csv(file5)

icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()


admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns



creatinine_itemid = d_items[d_items["label"] == "Creatinine (serum)"]["itemid"].values[0]


creatinine_events = chartevents[chartevents["itemid"] == creatinine_itemid]
creatinine_events = creatinine_events[["subject_id", "valuenum"]]
creatinine_events.rename(columns={"valuenum": "creatinine_level"}, inplace=True)

creatinine_events = creatinine_events[(creatinine_events["creatinine_level"] > 0) &
                                      (creatinine_events["creatinine_level"] <= 20)]

creatinine_means = creatinine_events.groupby("subject_id")["creatinine_level"].median().reset_index()
creatinine_means.rename(columns={"creatinine_level": "mean_creatinine"}, inplace=True)

merged_data = pd.merge(creatinine_means, merged_data, on="subject_id", how="inner")
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

merged_data["hospital_expire_flag"] = merged_data["hospital_expire_flag"].astype(int)


palette = {"0": "#A7C7E7", "1": "#F4A7B9"}

sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 8))
sns.boxplot(
    data=merged_data,
    x="hospital_expire_flag",
    y="mean_creatinine",
    palette=palette,
    width=0.6,
    linewidth=2
)

plt.title("Comparison of Mean Creatinine Levels by Mortality", fontsize=18, fontweight="bold", color="darkblue")
plt.xlabel("Mortality Status", fontsize=14, labelpad=10, color="darkblue")
plt.ylabel("Mean Creatinine Levels (mg/dL)", fontsize=14, labelpad=10, color="darkblue")
plt.xticks(ticks=[0, 1], labels=["Survived", "Deceased"], fontsize=12, color="darkblue")

sns.stripplot(
    data=merged_data,
    x="hospital_expire_flag",
    y="mean_creatinine",
    size=6,
    color="gray",
    alpha=0.5,
    jitter=True
)


plt.grid(visible=True, linestyle="--", linewidth=0.7, alpha=0.6)


plt.tight_layout()


plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


file_path1 = r"/content/icustays.csv"
file_path2 = r"/content/patients.csv"
file_path3 = r"/content/d_items.csv"
file_path4 = r"/content/inputevents.csv"


icustays = pd.read_csv(file_path1)
patients = pd.read_csv(file_path2)
d_items = pd.read_csv(file_path3)
inputevents = pd.read_csv(file_path4)



icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

norepinephrine_itemid = d_items[d_items["label"] == "Norepinephrine"]["itemid"].values[0]


norepinephrine_events = inputevents[inputevents["itemid"] == norepinephrine_itemid]
norepinephrine_events = norepinephrine_events[["subject_id", "amount"]]
norepinephrine_events.rename(columns={"amount": "norepinephrine_value"}, inplace=True)

norepinephrine_events = norepinephrine_events[(norepinephrine_events["norepinephrine_value"] > 0) &
                                              (norepinephrine_events["norepinephrine_value"] <= 100)]

norepinephrine_means = norepinephrine_events.groupby("subject_id")["norepinephrine_value"].median().reset_index()
norepinephrine_means.rename(columns={"norepinephrine_value": "mean_norepinephrine"}, inplace=True)


merged_data = pd.merge(norepinephrine_means, merged_data, on="subject_id", how="inner")


merged_data["hospital_expire_flag"] = merged_data["hospital_expire_flag"].astype(int)


palette = {"0": "#A7C7E7", "1": "#F4A7B9"}

sns.set_theme(style="whitegrid")


plt.figure(figsize=(12, 8))
sns.boxplot(
    data=merged_data,
    x="hospital_expire_flag",
    y="mean_norepinephrine",
    palette=palette,
    width=0.6,
    linewidth=2
)


plt.title("Comparison of Mean Norepinephrine Levels by Mortality", fontsize=18, fontweight="bold", color="darkblue")
plt.xlabel("Mortality Status", fontsize=14, labelpad=10, color="darkblue")
plt.ylabel("Mean Norepinephrine Levels (mcg/min)", fontsize=14, labelpad=10, color="darkblue")
plt.xticks(ticks=[0, 1], labels=["Survived", "Deceased"], fontsize=12, color="darkblue")


sns.stripplot(
    data=merged_data,
    x="hospital_expire_flag",
    y="mean_norepinephrine",
    size=6,
    color="gray",
    alpha=0.5,
    jitter=True
)

plt.grid(visible=True, linestyle="--", linewidth=0.7, alpha=0.6)


plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


file_path1 = r"/content/icustays.csv"
file_path2 = r"/content/patients.csv"
file_path3 = r"/content/d_items.csv"
file_path4 = r"/content/inputevents.csv"


icustays = pd.read_csv(file_path1)
patients = pd.read_csv(file_path2)
d_items = pd.read_csv(file_path3)
inputevents = pd.read_csv(file_path4)



icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),
    icu_stay_count=("hadm_id", "count")
).reset_index()

admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]

merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Dextrose_5_percent_itemid  = d_items[d_items["label"] == "Dextrose 5%"]["itemid"].values[0]


Dextrose_5_percent_events = inputevents[inputevents["itemid"] == Dextrose_5_percent_itemid]
Dextrose_5_percent_events =Dextrose_5_percent_events[["subject_id", "amount"]]
Dextrose_5_percent_events.rename(columns={"amount": "Dextrose_5%_value"}, inplace=True)

Dextrose_5_percent_events = Dextrose_5_percent_events[(Dextrose_5_percent_events["Dextrose_5%_value"] > 0) &
                                              (Dextrose_5_percent_events["Dextrose_5%_value"] <= 100)]

Dextrose_5_percent_means = Dextrose_5_percent_events.groupby("subject_id")["Dextrose_5%_value"].median().reset_index()
Dextrose_5_percent_means.rename(columns={"Dextrose_5%_value": "mean_Dextrose_5%"}, inplace=True)


merged_data = pd.merge(Dextrose_5_percent_means, merged_data, on="subject_id", how="inner")


merged_data["hospital_expire_flag"] = merged_data["hospital_expire_flag"].astype(int)


palette = {"0": "#A7C7E7", "1": "#F4A7B9"}

sns.set_theme(style="whitegrid")


plt.figure(figsize=(12, 8))
sns.boxplot(
    data=merged_data,
    x="hospital_expire_flag",
    y="mean_Dextrose_5%",
    palette=palette,
    width=0.6,
    linewidth=2
)


plt.title("Comparison of Mean Dextrose_5% Levels by Mortality", fontsize=18, fontweight="bold", color="darkblue")
plt.xlabel("Mortality Status", fontsize=14, labelpad=10, color="darkblue")
plt.ylabel("Mean Dextrose_5% Levels (mcg/min)", fontsize=14, labelpad=10, color="darkblue")
plt.xticks(ticks=[0, 1], labels=["Survived", "Deceased"], fontsize=12, color="darkblue")


sns.stripplot(
    data=merged_data,
    x="hospital_expire_flag",
    y="mean_Dextrose_5%",
    size=6,
    color="gray",
    alpha=0.5,
    jitter=True
)

plt.grid(visible=True, linestyle="--", linewidth=0.7, alpha=0.6)


plt.tight_layout()

plt.show()

In [ ]:
file_path1 = r"/content/icustays.csv"
file_path2 = r"/content/patients.csv"
file_path3 = r"/content/d_items.csv"
file_path4 = r"/content/inputevents.csv"

icustays = pd.read_csv(file_path1)
patients = pd.read_csv(file_path2)
d_items = pd.read_csv(file_path3)
inputevents = pd.read_csv(file_path4)

# Extract Dextrose 5% item ID
Dextrose_5_percent_itemid = d_items[d_items["label"] == "Dextrose 5%"]["itemid"].values[0]

# Filter events related to Dextrose 5%
Dextrose_5_percent_events = inputevents[inputevents["itemid"] == Dextrose_5_percent_itemid]
Dextrose_5_percent_events = Dextrose_5_percent_events[["subject_id", "amount"]]
Dextrose_5_percent_events.rename(columns={"amount": "Dextrose_5%_value"}, inplace=True)

# Filtering outliers for further analysis
outliers = Dextrose_5_percent_events[
    (Dextrose_5_percent_events["Dextrose_5%_value"] > 100) |
    (Dextrose_5_percent_events["Dextrose_5%_value"] <= 0)
]

# Merging the outliers with mortality data
outliers_summary = outliers.merge(merged_data[["subject_id", "hospital_expire_flag"]], on="subject_id", how="inner")

# Counting outliers for each mortality status
outliers_count = outliers_summary.groupby("hospital_expire_flag")["subject_id"].count().reset_index()
outliers_count.rename(columns={"subject_id": "outlier_count", "hospital_expire_flag": "Mortality Status"}, inplace=True)

# Mapping mortality labels
outliers_count["Mortality Status"] = outliers_count["Mortality Status"].map({0: "Survived", 1: "Deceased"})

# Plotting the bar chart
plt.figure(figsize=(10, 6))
sns.barplot(data=outliers_count, x="Mortality Status", y="outlier_count", palette="coolwarm", edgecolor="black")

# Customizing the plot
plt.title("Outliers in Dextrose 5% Levels by Mortality Status", fontsize=18, fontweight="bold", color="darkblue")
plt.xlabel("Mortality Status", fontsize=14, labelpad=10, color="darkblue")
plt.ylabel("Number of Outliers", fontsize=14, labelpad=10, color="darkblue")
plt.xticks(fontsize=12, color="darkblue")
plt.yticks(fontsize=12, color="darkblue")
plt.grid(axis='y', linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/ingredientevents.csv"
file_path11 = r"/content/patients.csv"
file_path7 = r"/content/d_items.csv"



icustays = pd.read_csv(file_path2)
ingredientevents = pd.read_csv(file_path4)
patients = pd.read_csv(file_path11)
d_items = pd.read_csv(file_path7)


# Aggregate icustays data
icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),  # Average length of stay
    icu_stay_count=("hadm_id", "count")  # Number of ICU stays
).reset_index()

# Ensure admittime and dischtime are datetime for proper sorting
admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

# Aggregate admissions data for the last admission per patient
admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]

# Merge data into a single dataset (focus on last admission data)
merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

# Separate data for alive and deceased patients
alive_patients = merged_data[merged_data['hospital_expire_flag'] == 0][['subject_id']]
deceased_patients = merged_data[merged_data['hospital_expire_flag'] == 1][['subject_id']]

# Filter ingredient events for both categories
alive_ingredientevents = pd.merge(alive_patients, ingredientevents, on='subject_id', how='inner')
deceased_ingredientevents = pd.merge(deceased_patients, ingredientevents, on='subject_id', how='inner')

# Count the frequency of each item ID for alive and deceased patients
alive_item_counts = alive_ingredientevents['itemid'].value_counts().reset_index()
deceased_item_counts = deceased_ingredientevents['itemid'].value_counts().reset_index()

alive_item_counts.columns = ['itemid', 'alive_count']
deceased_item_counts.columns = ['itemid', 'deceased_count']
# Merge the counts with ingredient names
alive_item_counts = pd.merge(alive_item_counts, d_items[['itemid', 'label']], on='itemid', how='left')
deceased_item_counts = pd.merge(deceased_item_counts, d_items[['itemid', 'label']], on='itemid', how='left')

# Rename the 'label' column to 'ingredient_name' for consistency
alive_item_counts.rename(columns={'label': 'ingredient_name'}, inplace=True)
deceased_item_counts.rename(columns={'label': 'ingredient_name'}, inplace=True)

# Merge alive and deceased counts
combined_counts = pd.merge(alive_item_counts, deceased_item_counts, on=['itemid', 'ingredient_name'], how='outer', suffixes=('_alive', '_deceased'))
combined_counts.fillna(0, inplace=True)

# Sort by combined count and take top 10 items
combined_counts['total_count'] = combined_counts['alive_count'] + combined_counts['deceased_count']
top_10_items = combined_counts.nlargest(10, 'total_count')
import seaborn as sns
import matplotlib.pyplot as plt

# Prepare the data for Seaborn
melted_data = top_10_items.melt(
    id_vars=['ingredient_name'],
    value_vars=['alive_count', 'deceased_count'],
    var_name='Status',
    value_name='Count'
)

# Map the status for better readability
melted_data['Status'] = melted_data['Status'].map({
    'alive_count': 'Alive',
    'deceased_count': 'Deceased'
})

# Create the plot
plt.figure(figsize=(14, 8))
sns.set_theme(style="whitegrid")  # Use a clean theme
sns.barplot(
    data=melted_data,
    x='Count',
    y='ingredient_name',
    hue='Status',
    palette=['#87CEEB', '#FFB6C1'],  # Soft pastel colors for visual appeal
    edgecolor='black'  # Add black edges for clarity
)

# Add labels and title
plt.xlabel('Frequency', fontsize=14, labelpad=10)
plt.ylabel('Ingredient Name', fontsize=14, labelpad=10)
plt.title('Top 10 Most Frequent Ingredient Events for Alive and Deceased Patients', fontsize=16, pad=20)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title='Status', title_fontsize=13, fontsize=12, loc='upper right', frameon=True)

# Enhance layout
plt.tight_layout()
sns.despine(left=True, bottom=True)  # Remove unnecessary spines for a cleaner look

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/ingredientevents.csv"
file_path11 = r"/content/patients.csv"
file_path7 = r"/content/d_items.csv"

icustays = pd.read_csv(file_path2)
ingredientevents = pd.read_csv(file_path4)
patients = pd.read_csv(file_path11)
d_items = pd.read_csv(file_path7)

# Aggregate icustays data
icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),  # Average length of stay
    icu_stay_count=("hadm_id", "count")  # Number of ICU stays
).reset_index()

# Ensure admittime and dischtime are datetime for proper sorting
admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

# Aggregate admissions data for the last admission per patient
admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]

# Merge data into a single dataset (focus on last admission data)
merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

# Separate data for alive and deceased patients
alive_patients = merged_data[merged_data['hospital_expire_flag'] == 0][['subject_id']]
deceased_patients = merged_data[merged_data['hospital_expire_flag'] == 1][['subject_id']]

# Filter ingredient events for both categories
alive_ingredientevents = pd.merge(alive_patients, ingredientevents, on='subject_id', how='inner')
deceased_ingredientevents = pd.merge(deceased_patients, ingredientevents, on='subject_id', how='inner')

# Count the frequency of each item ID for alive and deceased patients
alive_item_counts = alive_ingredientevents['itemid'].value_counts().reset_index()
deceased_item_counts = deceased_ingredientevents['itemid'].value_counts().reset_index()

alive_item_counts.columns = ['itemid', 'alive_count']
deceased_item_counts.columns = ['itemid', 'deceased_count']
# Merge the counts with ingredient names
alive_item_counts = pd.merge(alive_item_counts, d_items[['itemid', 'label']], on='itemid', how='left')
deceased_item_counts = pd.merge(deceased_item_counts, d_items[['itemid', 'label']], on='itemid', how='left')

# Rename the 'label' column to 'ingredient_name' for consistency
alive_item_counts.rename(columns={'label': 'ingredient_name'}, inplace=True)
deceased_item_counts.rename(columns={'label': 'ingredient_name'}, inplace=True)

# Merge alive and deceased counts
combined_counts = pd.merge(alive_item_counts, deceased_item_counts, on=['itemid', 'ingredient_name'], how='outer', suffixes=('_alive', '_deceased'))
combined_counts.fillna(0, inplace=True)

# Sort by combined count and take top 10 items
combined_counts['total_count'] = combined_counts['alive_count'] + combined_counts['deceased_count']
top_10_items = combined_counts.nlargest(10, 'total_count')

# Prepare the data for Seaborn
melted_data = top_10_items.melt(
    id_vars=['ingredient_name'],
    value_vars=['alive_count', 'deceased_count'],
    var_name='Status',
    value_name='Count'
)

# Map the status for better readability
melted_data['Status'] = melted_data['Status'].map({
    'alive_count': 'Alive',
    'deceased_count': 'Deceased'
})

# Calculate overall percentages for alive and deceased patients
total_alive = melted_data[melted_data['Status'] == 'Alive']['Count'].sum()
total_deceased = melted_data[melted_data['Status'] == 'Deceased']['Count'].sum()

def calculate_overall_percentage(row):
    if row['Status'] == 'Alive':
        return (row['Count'] / total_alive) * 100
    else:
        return (row['Count'] / total_deceased) * 100

melted_data['Percentage'] = melted_data.apply(calculate_overall_percentage, axis=1)

# Adjust the count values to represent percentages
melted_data['Count'] = melted_data['Percentage']

# Create the plot
plt.figure(figsize=(14, 8))
sns.set_theme(style="whitegrid")  # Use a clean theme
barplot = sns.barplot(
    data=melted_data,
    x='Count',
    y='ingredient_name',
    hue='Status',
    palette=['#87CEEB', '#FFB6C1'],  # Soft pastel colors for visual appeal
    edgecolor='black'  # Add black edges for clarity
)

# Add labels and title
plt.xlabel('Percentage (%)', fontsize=14, labelpad=10)
plt.ylabel('Ingredient Name', fontsize=14, labelpad=10)
plt.title('Top 10 Most Frequent Ingredient Events for Alive and Deceased Patients (Percentage)', fontsize=16, pad=20)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title='Status', title_fontsize=13, fontsize=12, loc='upper right', frameon=True)

# Add percentage labels to each bar
for p, percentage in zip(barplot.patches, melted_data['Percentage']):
    barplot.annotate(f'{percentage:.1f}%',
                     (p.get_width() + 1, p.get_y() + p.get_height() / 2),
                     ha='center', va='center', fontsize=10, color='black', xytext=(5, 0),
                     textcoords='offset points')

# Enhance layout
plt.tight_layout()
sns.despine(left=True, bottom=True)  # Remove unnecessary spines for a cleaner look

# Show the plot
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

# Load the dataset
data = pd.read_csv(r"/content/testing_h.csv")

# Step 1: Define thresholds for extreme values
variables = [
    'norepinephrine_dosage', 'total_creatinine_value', 'blood_pressure_mean',
    'mean_glucose', 'mean_potassium', 'mean_sodium',
    'mean_arterial_co2', 'mean_ph_arterial', 'total_ventilation_duration', 'hrv_range',
    'mean_sao2', 'min_sao2', 'std_sao2', 'time_below_90','mean_ph_arterial'
]

# Define extreme thresholds for deceased patients
deceased_data = data[data['hospital_expire_flag'] == 1]
extreme_thresholds = {var: deceased_data[var].quantile(0.90) for var in variables}
low_thresholds = {var: deceased_data[var].quantile(0.10) for var in variables}

# Create binary flags for extreme values
for var in variables:
    if var in ['mean_ph_arterial', 'mean_arterial_co2', 'min_sao2','blood_pressure_mean','mean_sao2','mean_ph_arterial']:  # Low is critical
        flag_name = f'low_{var}'
        data[flag_name] = (data[var] < low_thresholds[var]).astype(int)
    else:  # High is critical
        flag_name = f'high_{var}'
        data[flag_name] = (data[var] > extreme_thresholds[var]).astype(int)

# Step 2: Create new interaction features
data['glucose_potassium_interaction'] = data['mean_glucose'] * data['mean_potassium']
data['norepinephrine_pressure_interaction'] = data['high_norepinephrine_dosage'] * data['blood_pressure_mean']
data['sao2_variability_interaction'] = data['std_sao2'] * data['time_below_90']
data['metabolic_stress(mean_sao2 X mean_glucose)'] = data['mean_sao2'] * data['mean_glucose']

# Step 3: Define features and target
features = [
    'high_norepinephrine_dosage',
    'blood_pressure_mean', 'high_mean_glucose', 'high_mean_potassium', 'high_mean_sodium',
    'high_total_ventilation_duration',
    'high_hrv_range',
    'glucose_potassium_interaction','mean_sao2', 'min_sao2', 'std_sao2', 'time_below_90',
    'metabolic_stress(mean_sao2 X mean_glucose)','mean_ph_arterial'
]
target = 'hospital_expire_flag'

X = data[features]
y = data[target]

# Step 4: Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


from imblearn.over_sampling import SMOTE
# Step 4.1: Apply SMOTE to oversample the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


# Step 5: Standardize the features
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)


# Step 6: Train a logistic regression model
logistic_model = LogisticRegression(max_iter=1000, class_weight='balanced')
logistic_model.fit(X_train_resampled, y_train_resampled)


# Step 7: Evaluate the model
y_pred = logistic_model.predict(X_test)
y_proba = logistic_model.predict_proba(X_test)[:, 1]


# Performance metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

# Feature importance
coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': logistic_model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)
print("\nFeature Coefficients:\n", coefficients)

# Plot feature coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=coefficients, palette='coolwarm')
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

# Step 8: Correlation heatmap
correlation_matrix = data[features + [target]].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar=True,
    square=True
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()



from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, accuracy_score

# Step 6: Train a linear regression model
linear_model = LinearRegression()
linear_model.fit(X_train_resampled, y_train_resampled)

# Step 7: Predict using the linear regression model
y_lr_pred = linear_model.predict(X_test)

# Step 8: Convert predictions to binary outcomes
threshold = 0.5  # Standard threshold for binary classification
y_lr_binary_pred = (y_lr_pred >= threshold).astype(int)

# Step 9: Evaluate the linear regression model
mse = mean_squared_error(y_test, y_lr_pred)
r2 = r2_score(y_test, y_lr_pred)
roc_auc = roc_auc_score(y_test, y_lr_pred)

# Print evaluation metrics
print("Linear Regression Mean Squared Error:", mse)
print("Linear Regression R^2 Score:", r2)
print("Linear Regression ROC-AUC:", roc_auc)
print("Linear Regression Accuracy (binary threshold = 0.5):", accuracy_score(y_test, y_lr_binary_pred))

# Step 10: Analyze coefficients
lr_coefficients = pd.DataFrame({
    'Feature': features,
    'Coefficient': linear_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print("\nLinear Regression Coefficients:\n", lr_coefficients)

# Plot feature coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=lr_coefficients, palette='coolwarm')
plt.title("Linear Regression Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


file_path1 = r"/content/admissions.csv"
file_path2 = r"/content/icustays.csv"
file_path4 = r"/content/patients.csv"
file_path11 = r"/content/chartevents.csv"
file_path7 = r"/content/d_items.csv"


admissions = pd.read_csv(file_path1)
icustays = pd.read_csv(file_path2)
chartevents = pd.read_csv(file_path11)
patients = pd.read_csv(file_path4)
d_items = pd.read_csv(file_path7)


# Aggregate icustays data
icustays_summary = icustays.groupby("subject_id").agg(
    avg_los=("los", "mean"),  # Average length of stay
    icu_stay_count=("hadm_id", "count")  # Number of ICU stays
).reset_index()

# Ensure admittime and dischtime are datetime for proper sorting
admissions["admittime"] = pd.to_datetime(admissions["admittime"], errors="coerce")
admissions["dischtime"] = pd.to_datetime(admissions["dischtime"], errors="coerce")

# Aggregate admissions data for the last admission per patient
admissions_summary = admissions.sort_values(by=["subject_id", "admittime"]).groupby("subject_id").last().reset_index()
admissions_summary = admissions_summary[["subject_id", "hadm_id", "admittime", "dischtime", "hospital_expire_flag"]]

# Merge data into a single dataset (focus on last admission data)
merged_data = (
    patients.merge(icustays_summary, on="subject_id", how="left")
            .merge(admissions_summary, on="subject_id", how="left")
)

# Separate data for alive and deceased patients

import seaborn as sns
import matplotlib.pyplot as plt

# Filter for glucose measurements (itemid = 220621)
glucose_data = chartevents[chartevents['itemid'] == 220621]

# Merge glucose data with patient information (alive vs deceased)
glucose_data = pd.merge(glucose_data, merged_data[['subject_id', 'hospital_expire_flag']], on='subject_id', how='inner')

# Calculate mean glucose level per patient
glucose_means = glucose_data.groupby(['subject_id', 'hospital_expire_flag'])['valuenum'].mean().reset_index()
glucose_means.rename(columns={'valuenum': 'mean_glucose'}, inplace=True)

# Map alive/deceased for better readability
glucose_means['Status'] = glucose_means['hospital_expire_flag'].map({0: 'Alive', 1: 'Deceased'})

# Remove outliers using the interquartile range (IQR)
Q1 = glucose_means['mean_glucose'].quantile(0.25)
Q3 = glucose_means['mean_glucose'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out extreme values
filtered_data = glucose_means[
    (glucose_means['mean_glucose'] >= lower_bound) &
    (glucose_means['mean_glucose'] <= upper_bound)
]

# Create the improved box plot
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")
sns.boxplot(
    data=filtered_data,
    x='Status',
    y='mean_glucose',
    palette=['lightblue', 'lightcoral'],
    width=0.6
)

# Add mean points
sns.stripplot(
    data=filtered_data,
    x='Status',
    y='mean_glucose',
    color='black',
    alpha=0.5,
    jitter=True
)

# Add title and labels
plt.title('Mean Glucose Levels for Alive vs Deceased Patients (Without Outliers)', fontsize=16, pad=20)
plt.xlabel('Patient Status', fontsize=14, labelpad=10)
plt.ylabel('Mean Glucose Level', fontsize=14, labelpad=10)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Display the plot
plt.tight_layout()
plt.show()
